<a href="https://colab.research.google.com/github/papy-studio/Coding_week_Groupe-17/blob/mbk/Gradient_Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import shap

In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [ ]:
import sys

!pip install ucimlrepo

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

# Charger la data originale pour comparer
dataset = fetch_ucirepo(id=544)
df_original = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df = df_original.copy()

# ── AVANT ────────────────────────────────────────────────────────────────────
print("=" * 50)
print("AVANT — Data originale")
print("=" * 50)
print(df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].head(3))
print(f"\nTypes : \n{df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].dtypes}")

# ── ENCODAGE ─────────────────────────────────────────────────────────────────
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

le = LabelEncoder()
for col in ['Gender', 'CAEC', 'CALC', 'MTRANS']:
    df[col] = le.fit_transform(df[col])

le_target = LabelEncoder()
df['NObeyesdad'] = le_target.fit_transform(df['NObeyesdad'])

# ── APRÈS ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("APRÈS — Data encodée")
print("=" * 50)
print(df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].head(3))
print(f"\nTypes : \n{df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].dtypes}")

# ── MÉMOIRE ───────────────────────────────────────────────────────────────────
def optimize_memory(df):
    before = df.memory_usage(deep=True).sum() / 1024
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    after = df.memory_usage(deep=True).sum() / 1024
    print(f"\n{'='*50}")
    print("OPTIMISATION MÉMOIRE")
    print(f"{'='*50}")
    print(f"Avant  : {before:.1f} KB")
    print(f"Après  : {after:.1f} KB")
    print(f"Gagné  : {((before-after)/before*100):.1f}%")
    return df

df = optimize_memory(df)

# ── SPLIT ─────────────────────────────────────────────────────────────────────
X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n{'='*50}")
print("SÉPARATION TRAIN / TEST")
print(f"{'='*50}")
print(f"Total lignes    : {len(df)}")
print(f"Train (80%)     : {X_train.shape[0]} lignes")
print(f"Test  (20%)     : {X_test.shape[0]} lignes")
print(f"\n✅ Data prête pour le ML !")


AVANT — Data originale
   Gender SMOKE       CAEC     NObeyesdad
0  Female    no  Sometimes  Normal_Weight
1  Female   yes  Sometimes  Normal_Weight
2    Male    no  Sometimes  Normal_Weight

Types : 
Gender        object
SMOKE         object
CAEC          object
NObeyesdad    object
dtype: object

APRÈS — Data encodée
   Gender  SMOKE  CAEC  NObeyesdad
0       0      0     2           1
1       0      1     2           1
2       1      0     2           1

Types : 
Gender        int64
SMOKE         int64
CAEC          int64
NObeyesdad    int64
dtype: object

OPTIMISATION MÉMOIRE
Avant  : 280.5 KB
Après  : 140.3 KB
Gagné  : 50.0%

SÉPARATION TRAIN / TEST
Total lignes    : 2111
Train (80%)     : 1688 lignes
Test  (20%)     : 423 lignes

✅ Data prête pour le ML !


In [6]:
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

print("Data splits saved to 'data/' directory:")
print('data/X_train.csv')
print('data/X_test.csv')
print('data/y_train.csv')
print('data/y_test.csv')

Data splits saved to 'data/' directory:
data/X_train.csv
data/X_test.csv
data/y_train.csv
data/y_test.csv


In [9]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize the Gradient Boosting Classifier
gbc_model = GradientBoostingClassifier(random_state=42)

# Train the model
gbc_model.fit(X_train, y_train)

print("Modèle GradientBoostingClassifier entraîné avec succès.")

Modèle GradientBoostingClassifier entraîné avec succès.


In [18]:
# Make predictions on the test set with the GBC model
y_pred_gbc = gbc_model.predict(X_test)

# Evaluate the GBC model's accuracy
accuracy_gbc = accuracy_score(y_test, y_pred_gbc)

print(f"Précision du modèle Gradient Boosting sur l'ensemble de test : {accuracy_gbc:.4f}")

NameError: name 'accuracy_score' is not defined

In [12]:
# Confusion Matrix for GradientBoostingClassifier
cm_gbc = confusion_matrix(y_test, y_pred_gbc)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_gbc, annot=True, fmt='d', cmap='Greens')
plt.title('Matrice de Confusion pour GradientBoostingClassifier')
plt.xlabel('Prédit')
plt.ylabel('Vrai')
plt.show()

NameError: name 'confusion_matrix' is not defined

In [13]:
import pandas as pd

# Define the feature order as expected by the model
feature_columns = ['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight',
                   'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE',
                   'CALC', 'MTRANS']

# Inverse transform mapping for NObeyesdad
# Assuming le_target is still available from cell z7dFXascnHP6
obesity_classes = le_target.classes_

def predict_obesity_class(patient_data):
    """
    Prédit la classe d'obésité d'un patient.

    Args:
        patient_data (dict): Un dictionnaire contenant les informations du patient.
                             Les clés doivent correspondre aux noms des colonnes dans feature_columns.
                             Les valeurs catégorielles doivent être déjà encodées numériquement.

    Returns:
        str: La classe d'obésité prédite sous forme de chaîne de caractères.
    """
    # Create a DataFrame from the patient data
    patient_df = pd.DataFrame([patient_data])

    # Ensure the columns are in the correct order
    patient_df = patient_df[feature_columns]

    # Make prediction using the GBC model
    prediction = gbc_model.predict(patient_df)

    # Convert the numerical prediction back to original class name
    predicted_class = obesity_classes[prediction[0]]

    return predicted_class

print("Fonction 'predict_obesity_class' définie.")

Fonction 'predict_obesity_class' définie.


### Exemple d'utilisation de la fonction de prédiction

Créons un exemple de patient avec des données fictives en utilisant les encodages numériques que nous avons définis ci-dessus.

In [27]:
# Exemple de données pour un nouveau patient (toutes les valeurs sont des exemples et doivent être ajustées)
# Rappel des encodages numériques:
# Gender: 0=Female, 1=Male
# family_history_with_overweight, FAVC, SMOKE, SCC: 0=no, 1=yes
# CAEC, CALC: 0=Always, 1=Frequently, 2=Sometimes, 3=no
# MTRANS: 0=Automobile, 1=Bike, 2=Motorbike, 3=Public_Transportation, 4=Walking

new_patient_info = {
    'Gender': 0,  # Female
    'Age': 25.0,
    'Height': 1.65, # meters
    'Weight': 68.0, # kg
    'family_history_with_overweight': 1, # yes
    'FAVC': 1, # yes
    'FCVC': 2.0, # e.g., 'Sometimes' equivalent for a numerical scale
    'NCP': 3.0, # e.g., 3 main meals
    'CAEC': 2, # Sometimes
    'SMOKE': 0, # no
    'CH2O': 2.0, # e.g., 2 liters
    'SCC': 0, # no
    'FAF': 1.0, # e.g., 1 time per week
    'TUE': 1.0, # e.g., 1 hour per day
    'CALC': 2, # Sometimes
    'MTRANS': 3 # Public_Transportation
}

predicted_obesity = predict_obesity_class(new_patient_info)

print(f"Pour le patient donné, la classe d'obésité prédite est : {predicted_obesity}")

Pour le patient donné, la classe d'obésité prédite est : Normal_Weight


In [17]:
import shap

explainer = shap.TreeExplainer(gbc_model, X_train)

TypeError: The passed model is not callable and cannot be analyzed directly with the given masker! Model: GradientBoostingClassifier(random_state=42)

In [1]:
shap_values = explainer(X_test, check_additivity=False)

NameError: name 'explainer' is not defined

In [7]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")

y_train = pd.read_csv("data/y_train.csv")
y_test = pd.read_csv("data/y_test.csv")

model = LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6)

model.fit(X_train, y_train.values.ravel())

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")
roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr")

print("Accuracy :", accuracy)
print("Precision :", precision)
print("Recall :", recall)
print("F1-score :", f1)
print("ROC-AUC :", roc_auc)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000650 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2063
[LightGBM] [Info] Number of data points in the train set: 1688, number of used features: 16
[LightGBM] [Info] Start training from score -2.056021
[LightGBM] [Info] Start training from score -2.015199
[LightGBM] [Info] Start training from score -1.821828
[LightGBM] [Info] Start training from score -1.954836
[LightGBM] [Info] Start training from score -1.866779
[LightGBM] [Info] Start training from score -1.975979
[LightGBM] [Info] Start training from score -1.950661
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p